## Shrnutí

Tým logistických operací plánuje randomizovaný polní pokus porovnávající tři strategie směrování řidičů (statická základní linie, dynamické přesměrování, AI-optimalizované), napříč třemi regiony dep, s průměrným zpožděním doručení (v minutách) jako odezvou. PROC GLMPOWER vezme *příkladný* dataset předpokládaných průměrů buněk a řeší celkový počet řidičů potřebných k detekci efektů strategie při síle 80 % a 90 %, poté mapuje, jak dosažitelná síla ubývá s rostoucí variabilitou mezi trasami.

# Stanovení velikosti polního pokusu směrování řidičů pomocí PROC GLMPOWER

## Shrnutí

Tým logistických operací se chystá spustit randomizovaný polní pokus porovnávající tři strategie směrování řidičů -- základní linii **Static**, systém přesměrování **Dynamic** a plánovač **AI-Optimized** -- nasazené napříč třemi regiony dep (Northeast, Midwest, West). Odezvou je průměrné **zpoždění doručení v minutách**. Než tým zaváže kapacitu flotily do pokusu, musí zodpovědět: *kolik řidičů potřebujeme, abychom spolehlivě detekovali očekávané provozní zlepšení?*

Tento notebook používá **PROC GLMPOWER** k provedení prospektivní analýzy síly a velikosti vzorku pro obecný lineární model stojící za pokusem. S využitím *příkladného* datasetu předpokládaných průměrů buněk (1) řeší celkový počet zapojených řidičů potřebný k dosažení 80% a 90% síly pro souhrnné efekty strategie a regionu, (2) mapuje, jak se dosažitelná síla zhoršuje s rostoucí variabilitou trasa-od-trasy, a (3) vytváří křivku síly na podporu rozhodnutí o zápisu.

> **Klíčové zjištění:** Při přijatelné směrodatné odchylce chyby 8 minut dodá zhruba **63 řidičů** 80% sílu a **83 řidičů** 90% sílu pro detekci efektů strategie směrování -- ale pokud se variabilita zpoždění blíží 10 minutám, ani 90 zapsaných řidičů nedosáhne 70% síly, což podtrhuje hodnotu úzkých, dobře instrumentovaných tras.

---
## 1. Sestavení příkladného designu

Prvním krokem je zakódování rozvržení pokusu a *předpokládaného* průměrného zpoždění týmu pro každou kombinaci strategie směrování x regionu depa. Fixujeme náhodné seed a přidáváme zanedbatelný šum, aby průměry vypadaly realisticky, zatímco vyvážená struktura 3x3 zůstává zachována. Váha `cell_n` rovna 1 v každé buňce říká GLMPOWER, že design je vyvážený.

In [1]:
/* ================================================================
   Vygenerování příkladného datasetu předpokládaných průměrných
   zpoždění.
   Jeden řádek na kombinaci strategie směrování x region depa.
   ================================================================ */
data routing_trial;
   CALL streaminit(20260531);
   DÉLKA routing_strategy $8 depot_region $9;
   ŠTÍTEK routing_strategy = "Strategie směrování"
         depot_region     = "Region depa"
         mean_delay_min   = "Průměrné zpoždění (min)"
         cell_n           = "Váha buňky";
   POLE strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   POLE region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   POLE smean[3]     _temporary_ (18.0 14.5 11.0);   /* předpokládané průměry strategií */
   POLE radj[3]      _temporary_ (1.5  0.0 -1.0);    /* regionální úpravy (min)  */
   OPAKUJ i = 1 TO 3;
      OPAKUJ j = 1 TO 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         VÝSTUP;
      KONEC;
   KONEC;
   ODSTRANIT i j;
SPUSTIT;

PROCEDURA TISK data=routing_trial ŠTÍTEK noobs;
   PROMĚNNÁ routing_strategy depot_region mean_delay_min cell_n;
   NÁZEV "Příkladné průměry buněk: předpokládané zpoždění doručení (minuty)";
SPUSTIT;


                           Příkladné průměry buněk: předpokládané zpoždění doručení (minuty)                            

   Strategie směrování  Region depa        Průměrné zpoždění (min)    Váha buňky
Static                  Northeast                     19.687507651             1
Static                  Midwest                      17.9871067398             1
Static                  West                         16.8240210086             1
Dynamic                 Northeast                    15.9537535365             1
Dynamic                 Midwest                      14.4415169858             1
Dynamic                 West                         12.8636389493             1
AIOpt                   Northeast                    12.6143811724             1
AIOpt                   Midwest                       10.893885771             1
AIOpt                   West                           9.635351403             1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Velikost vzorku pro souhrnný design

S hotovým designem požádáme GLMPOWER, aby **vyřešil celkovou velikost vzorku** (`NTOTAL = .`) při 80% a 90% síle. Příkaz `MODEL` specifikuje dvoucestné rozvržení s interakcí (`routing_strategy*depot_region`), `WEIGHT cell_n` deklaruje profil vyvážených vah a `STDDEV = 8` je předpokládaná směrodatná odchylka zpoždění doručení (root mean square error). `NFRACTIONAL` nechá proceduru hlásit přesné zlomkové počty před zaokrouhlením nahoru.

Také předregistrujeme tři plánovaná srovnání `CONTRAST` -- AI-Opt vs. Static, Dynamic vs. Static a jakékoli přesměrování vs. Static -- které dokumentují provozně smysluplné hypotézy, jež má pokus otestovat.

In [2]:
/* ================================================================
   Vyřešení celkového počtu řidičů potřebného k dosažení 80% a
   90% síly pro efekty strategie směrování a regionu depa.
   ================================================================ */
PROCEDURA glmpower data=routing_trial;
   TŘÍDA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   VÁHA cell_n;
   CONTRAST "AI-Opt vs. Static"      routing_strategy -1  0  1;
   CONTRAST "Dynamic vs. Static"     routing_strategy -1  1  0;
   CONTRAST "Jakékoli přesměrování vs. Static" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   NÁZEV "Velikost vzorku pro detekci efektů strategie směrování na zpoždění";
SPUSTIT;


                           Příkladné průměry buněk: předpokládané zpoždění doručení (minuty)                            

The GLMPOWER Procedure


             Fixed Scenario Elements             

Item                Value                        
------------------  -----------------------------
Dependent Variable  Průměrné zpoždění (min)      
Source              ROUTING_STRATEGY             
Source              DEPOT_REGION                 
Source              ROUTING_STRATEGY*DEPOT_REGION

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Citlivost síly na variabilitu a velikost pokusu

Odpověď o velikosti vzorku závisí na předpokládané směrodatné odchylce chyby, kterou dopředu zřídkakdy známe přesně. Zde obrátíme otázku: **fixujeme** několik kandidátských celkových zápisů (`NTOTAL = 45 90 180`) a **řešíme dosaženou sílu** (`POWER = .`) napříč sítí přijatelných směrodatných odchylek zpoždění (6, 8 a 10 minut). Výsledná tabulka je mapou citlivosti -- ukazuje, jak robustní je každý plán zápisu, pokud se reálná variabilita trasy ukáže vyšší, než jsme doufali.

In [3]:
/* ================================================================
   Síť citlivosti: dosažená síla napříč kandidátskými velikostmi
   pokusu a přijatelnými směrodatnými odchylkami zpoždění.
   ================================================================ */
PROCEDURA glmpower data=routing_trial;
   TŘÍDA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   VÁHA cell_n;
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   NÁZEV "Dosažená síla napříč scénáři variability a velikosti pokusu";
SPUSTIT;


                           Příkladné průměry buněk: předpokládané zpoždění doručení (minuty)                            

The GLMPOWER Procedure


             Fixed Scenario Elements             

Item                Value                        
------------------  -----------------------------
Dependent Variable  Průměrné zpoždění (min)      
Source              ROUTING_STRATEGY             
Source              DEPOT_REGION                 

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Křivka síly pro rozhodnutí o zápisu

Nakonec vysledujeme **křivku síly** -- dosaženou sílu při růstu celkového zápisu od 30 do 270 řidičů v krocích po 30 -- pro model strategie-plus-region při očekávané směrodatné odchylce 8 minut. Řešení `POWER = .` napříč touto sítí zápisu vytvoří křivku jako tabulovanou řadu `N Total` vs. `Power`, takže můžeme odečíst zápis, při kterém je překročen každý z konvenčních cílů 80 % a 90 %.

In [4]:
/* ================================================================
   Křivka síly: dosažená síla vs. celkový počet zapsaných řidičů,
   sledováno od 30 do 270 v krocích po 30 při očekávané
   směrodatné odchylce 8 minut.
   ================================================================ */
PROCEDURA glmpower data=routing_trial;
   TŘÍDA routing_strategy depot_region;
   MODEL mean_delay_min = routing_strategy depot_region;
   VÁHA cell_n;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   NÁZEV "Křivka síly: dosažená síla vs. celkový počet zapsaných řidičů";
SPUSTIT;


                           Příkladné průměry buněk: předpokládané zpoždění doručení (minuty)                            

The GLMPOWER Procedure


             Fixed Scenario Elements             

Item                Value                        
------------------  -----------------------------
Dependent Variable  Průměrné zpoždění (min)      
Source              ROUTING_STRATEGY             
Source              DEPOT_REGION                 

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interpretace a doporučení

Analýza dává provoznímu týmu obhajitelný plán zápisu:

- **Základní stanovení velikosti (oddíl 2).** Za předpokladu 8minutové reziduální směrodatné odchylky dosahuje plný dvoucestný model (strategie, region a jejich interakce) **80% síly při 63 řidičích** a **90% síly při 83 řidičích**. Se zaokrouhlením nahoru na útlum cíl zápisu blízko **90 řidičů** pohodlně překračuje práh 90 %.

- **Na citlivosti záleží (oddíl 3).** Síla je vysoce citlivá na variabilitu zpoždění. Při 90 řidičích klesá dosažená síla z ~99 % (SD=6) na ~87 % (SD=8) na ~68 % (SD=10). Pilotní studie s 45 řidiči je adekvátní pouze při nízké variabilitě (~81% síla při SD=6) a je vážně poddimenzovaná (~37 %), pokud SD dosáhne 10. Praktický důsledek: investice do konzistentních, dobře instrumentovaných tras k udržení nízké variability je stejně hodnotná jako přidávání řidičů.

- **Křivka síly (oddíl 4).** Vysledovaná pro model strategie-plus-region (bez interakčního členu, čočka použitá pro sondu citlivosti), dosažená síla stoupá z 0.37 při 30 řidičích na 0.69 při 60, 0.87 při 90 a 0.95 při 120, zplošťuje se nad 0.99 do 180. Při čtení křivky vůči cílům přichází 80% síla poblíž 77 řidičů a 90% poblíž 99 -- o něco výše než stanovení velikosti plného modelu v oddíle 2, protože vynechání interakčního členu rozprostírá stejný efekt na méně stupňů volnosti modelu.

**Doporučení:** Zapsat přibližně **90 řidičů** (30 na strategii směrování, vyváženě napříč třemi regiony dep). To překračuje 90% sílu pro plný model při očekávané 8minutové směrodatné odchylce, udržuje ~87% sílu i na konzervativnější křivce redukovaného modelu a udržuje pokus dostatečně malý na provedení v rámci jediného provozního čtvrtletí.

*Poznámka:* GLMPOWER analyzuje předpokládaný *design*, nikoli pozorované výsledky -- takže věrohodnost těchto čísel spočívá na předpokládaných průměrech a směrodatné odchylce. Týmy by měly revidovat stanovení velikosti, jakmile krátký pilotní projekt poskytne empirický odhad variability zpoždění doručení.